In [1]:
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# load dataset
games = pd.read_csv(r"C:\AnPhi\inst414\nba_dataset_inst414\Cleaned_Games.csv")
games.head()

,gameId,gameDate,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,gameType,attendance,arenaId,gameLabel,gameSubLabel,seriesGameNumber
0,42400407,6/22/2025 20:00,Oklahoma City,Thunder,1610612760,Indiana,Pacers,1610612754,103,91,1610612760,Playoffs,18203,1000052,NBA Finals,Game 7,7.0
1,42400406,6/19/2025 20:30,Indiana,Pacers,1610612754,Oklahoma City,Thunder,1610612760,108,91,1610612754,Playoffs,17274,1000063,NBA Finals,Game 6,6.0
2,42400405,6/16/2025 20:30,Oklahoma City,Thunder,1610612760,Indiana,Pacers,1610612754,120,109,1610612760,Playoffs,18203,1000052,NBA Finals,Game 5,5.0
3,42400404,6/13/2025 20:30,Indiana,Pacers,1610612754,Oklahoma City,Thunder,1610612760,104,111,1610612760,Playoffs,17274,1000063,NBA Finals,Game 4,4.0
4,42400403,6/11/2025 20:30,Indiana,Pacers,1610612754,Oklahoma City,Thunder,1610612760,116,107,1610612754,Playoffs,17274,1000063,NBA Finals,Game 3,3.0


In [18]:
games['gameDate'] = pd.to_datetime(games['gameDate'], format = '%m/%d/%Y %H:%M', errors='coerce')

# define seasons
def assign_seasons(date):
    year = date.year
    if date.month >= 10:
        return f"{year}-{year+1}"
    else:
        return f"{year-1}-{year}"

games['season'] = games['gameDate'].apply(assign_seasons)
games = games.sort_values(['season', 'gameDate']).reset_index(drop=True)

# home and away win pct per season calculation
games['homeSeasonWinPct'] = 0.0
games['awaySeasonWinPct'] = 0.0

# separate seasons
for season in games['season'].unique():
    season_games = games[games['season'] == season].copy()

    # tracking wins and games for each team
    team_wins = {}
    team_games = {}

    # season lists
    season_home_win_pct_list = []
    season_away_win_pct_list = []

    for idx, row in season_games.iterrows():
        home_team = row['hometeamId']
        away_team = row['awayteamId']

        # win pct
        home_pct = team_wins.get(home_team, 0) / max(team_games.get(home_team, 1), 1)
        away_pct = team_wins.get(away_team, 0) / max(team_games.get(away_team, 1), 1)

        season_home_win_pct_list.append(home_pct)
        season_away_win_pct_list.append(away_pct)

        # games played update
        team_games[home_team] = team_games.get(home_team, 0) + 1
        team_games[away_team] = team_games.get(away_team, 0) + 1

        # wins update
        if row['winner'] == home_team:
            team_wins[home_team] = team_wins.get(home_team, 0) + 1
        else:
            team_wins[away_team] = team_wins.get(away_team, 0) + 1

    # add columns to games by season
    season_indices = games[games['season'] == season].index
    games.loc[season_indices, 'homeSeasonWinPct'] = season_home_win_pct_list
    games.loc[season_indices, 'awaySeasonWinPct'] = season_away_win_pct_list

games.to_csv(
    r"C:\AnPhi\inst414\nba_dataset_inst414\Cleaned_Games_with_WinPct.csv", index=False
)

In [9]:
# create predictor variables
games['playoffs'] = games['gameType'].apply(lambda x: 1 if x == 'Playoffs' else 0)
games['homePlayoffInteraction'] = games['homeSeasonWinPct'] * games['playoffs']
games['homeWin'] = games.apply(lambda row: 1 if row['winner'] == row['hometeamId'] else 0, axis = 1)

# defining predictor and target variables
X = games[['homeSeasonWinPct', 'awaySeasonWinPct', 'playoffs', 'homePlayoffInteraction']]
y = games['homeWin']

In [10]:
# spliting dataset (70% training, 15% validation, 15% testing)
# 70% training and 30% temporary split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size = 0.30, random_state = 42
)

# 15% validation and 15% testing split
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size = 0.5, random_state = 42
)

print("Training size: ", len(X_train))
print("Validation size: ", len(X_val))
print("Test size: ", len(X_test))

Training size:  7970
Validation size:  1708
Test size:  1709


In [14]:
# initialize logistic regression model
start = time.time()

logmodel = LogisticRegression()

# train model on training dataset
model.fit(X_train, y_train)

end = time.time()

In [15]:
# tune the model
y_val_pred = model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print("Validation accuracy: ", val_accuracy)

Validation accuracy:  0.6364168618266979


In [16]:
# evaluate on test dataset
y_test_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print("Test Performance: ")
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("AUC: ", auc)
print("Iterations: ", model.n_iter_)
print("Convergence tolerance: ", model.tol)
print("Training time (seconds): ", end - start)

Test Performance: 
Accuracy:  0.6465769455822118
Precision:  0.6617526617526618
Recall:  0.8088088088088088
AUC:  0.6832501515600107
Iterations:  [6]
Convergence tolerance:  0.0001
Training time (seconds):  0.031168699264526367
